# Proyecto F1 — Predicción del Top 10 (prototipo en Colab)

**Ciencia de Datos, UTN FRM 2026 — Proyecto integrador**

Pregunta de investigación: ¿es posible predecir, antes del inicio de una
carrera de Fórmula 1, el ranking final de los diez primeros pilotos usando
exclusivamente información disponible antes de la carrera?

Este notebook es la etapa de **prototipo**: acá probamos contra la API real
de Jolpica-F1, miramos la forma de las respuestas, y validamos la lógica de
features antes de migrarla al paquete `f1/` y al DAG de Airflow.

**No es el entregable de la Entrega 1** — esa entrega exige Airflow corriendo
con una corrida en verde. Este notebook es el paso previo para llegar ahí
con la lógica ya probada.

## Decisión de diseño que hay que fijar ACÁ, antes de seguir

`final_position` es el objetivo. La pregunta trampa es si la posición de
largada (grid/qualy) cuenta como "antes de la carrera":

- **Opción A** (más estricta): solo datos anteriores al fin de semana de
  carrera. Sin grid.
- **Opción B** (más común en la práctica): el grid es información pública
  antes de que se apague el semáforo, así que se incluye.

Este notebook prototipa **ambas** por separado — la columna `grid_position`
queda calculada, y ustedes deciden en el análisis final si la usan como
feature o la descartan. Lo que no pueden hacer es usarla sin decidir y
documentar la elección.


In [ ]:
# Si corren esto en Colab, requests y pandas ya vienen instalados.
# Descomenten si hace falta:
# !pip install requests pandas -q

import json
import time
import requests
import pandas as pd

BASE_URL = "https://api.jolpi.ca/ergast/f1"
HEADERS = {"User-Agent": "utn-frm-ciencia-de-datos-proyecto-f1/1.0"}


## 1. Cliente mínimo de la API

Igual que en `f1/jolpica.py`: una sola función que pide y devuelve JSON
crudo, con reintentos. Todo lo demás se construye arriba de esto.


In [ ]:
def fetch_json(path: str, retries: int = 3, backoff: float = 1.5) -> dict:
    url = f"{BASE_URL}/{path.lstrip('/')}"
    ultimo_error = None
    for intento in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            ultimo_error = e
            time.sleep(backoff ** intento)
    raise RuntimeError(f"No se pudo bajar {url}: {ultimo_error}")


## 2. Primer contacto: ¿qué forma tiene realmente la respuesta?

Antes de asumir nada del esquema (Ergast clásico), pidan una temporada y
un resultado de carrera puntual, e impriman crudo. Esto casi seguro les va
a mostrar diferencias respecto de lo que documentamos "a ojo" — la fuente
real manda.


In [ ]:
calendario_2024 = fetch_json("2024.json?limit=100")
print(json.dumps(calendario_2024, indent=2)[:2000])


{
  "MRData": {
    "xmlns": "",
    "series": "f1",
    "url": "https://api.jolpi.ca/ergast/f1/2024.json",
    "limit": "100",
    "offset": "0",
    "total": "24",
    "RaceTable": {
      "season": "2024",
      "Races": [
        {
          "season": "2024",
          "round": "1",
          "url": "https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix",
          "raceName": "Bahrain Grand Prix",
          "Circuit": {
            "circuitId": "bahrain",
            "url": "https://en.wikipedia.org/wiki/Bahrain_International_Circuit",
            "circuitName": "Bahrain International Circuit",
            "Location": {
              "lat": "26.0325",
              "long": "50.5106",
              "locality": "Sakhir",
              "country": "Bahrain"
            }
          },
          "date": "2024-03-02",
          "time": "15:00:00Z",
          "FirstPractice": {
            "date": "2024-02-29",
            "time": "11:30:00Z"
          },
          "SecondPractice": {
   

In [ ]:
resultados_r1_2024 = fetch_json("2024/1/results.json?limit=100")
print(json.dumps(resultados_r1_2024, indent=2)[:2500])


{
  "MRData": {
    "xmlns": "",
    "series": "f1",
    "url": "https://api.jolpi.ca/ergast/f1/2024/1/results.json",
    "limit": "100",
    "offset": "0",
    "total": "20",
    "RaceTable": {
      "season": "2024",
      "round": "1",
      "Races": [
        {
          "season": "2024",
          "round": "1",
          "url": "https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix",
          "raceName": "Bahrain Grand Prix",
          "Circuit": {
            "circuitId": "bahrain",
            "url": "https://en.wikipedia.org/wiki/Bahrain_International_Circuit",
            "circuitName": "Bahrain International Circuit",
            "Location": {
              "lat": "26.0325",
              "long": "50.5106",
              "locality": "Sakhir",
              "country": "Bahrain"
            }
          },
          "date": "2024-03-02",
          "time": "15:00:00Z",
          "Results": [
            {
              "number": "1",
              "position": "1",
              

## 3. Aplanar un resultado de carrera a filas piloto-carrera

De acá sale la columna objetivo (`final_position`) y las columnas de
identificación. Ojo con los DNF/DNS: en Ergast/Jolpica, `position` no
existe como número cuando el piloto no completó la carrera — hay que
decidir cómo se codifica (lo dejamos como `None` acá, y con un chequeo
aparte de cuántos son).


In [ ]:
def resultados_a_filas(data_json: dict) -> list[dict]:
    races = data_json["MRData"]["RaceTable"]["Races"]
    if not races:
        return []
    race = races[0]
    filas = []
    for r in race["Results"]:
        posicion = int(r["position"]) if r.get("position", "").isdigit() else None
        filas.append({
            "season": int(race["season"]),
            "round": int(race["round"]),
            "race_id": f"{race['season']}_{race['round']}",
            "circuit_id": race["Circuit"]["circuitId"],
            "race_date": race["date"],
            "driver_id": r["Driver"]["driverId"],
            "constructor_id": r["Constructor"]["constructorId"],
            "grid_position": int(r["grid"]) if str(r.get("grid", "0")).isdigit() and r["grid"] != "0" else None,
            "final_position": posicion,
            "status": r.get("status"),  # "Finished", "Accident", "Engine", etc. -- para entender los nulos de final_position
        })
    return filas

filas_r1 = resultados_a_filas(resultados_r1_2024)
df_r1 = pd.DataFrame(filas_r1)
df_r1


,season,round,race_id,circuit_id,race_date,driver_id,constructor_id,grid_position,final_position,status
0,2024,1,2024_1,bahrain,2024-03-02,max_verstappen,red_bull,1,1,Finished
1,2024,1,2024_1,bahrain,2024-03-02,perez,red_bull,5,2,Finished
2,2024,1,2024_1,bahrain,2024-03-02,sainz,ferrari,4,3,Finished
3,2024,1,2024_1,bahrain,2024-03-02,leclerc,ferrari,2,4,Finished
4,2024,1,2024_1,bahrain,2024-03-02,russell,mercedes,3,5,Finished
5,2024,1,2024_1,bahrain,2024-03-02,norris,mclaren,7,6,Finished
6,2024,1,2024_1,bahrain,2024-03-02,hamilton,mercedes,9,7,Finished
7,2024,1,2024_1,bahrain,2024-03-02,piastri,mclaren,8,8,Finished
8,2024,1,2024_1,bahrain,2024-03-02,alonso,aston_martin,6,9,Finished
9,2024,1,2024_1,bahrain,2024-03-02,stroll,aston_martin,12,10,Finished


## 4. Standings ANTES de la carrera — acá es donde se cuela el leakage si no se tiene cuidado

Para la ronda `N`, el standing "antes de la carrera" es el standing
publicado tras la ronda `N-1`. Para la ronda 1 de una temporada no hay
standing previo (arranca de cero, salvo que decidan arrastrar el campeonato
del año anterior como feature — otra decisión de diseño para documentar).


In [ ]:
def driver_standings_before(season: int, round_: int) -> pd.DataFrame:
    ronda_previa = round_ - 1
    if ronda_previa < 1:
        return pd.DataFrame(columns=["driver_id", "driver_points_before", "driver_standing_before"])
    data = fetch_json(f"{season}/{ronda_previa}/driverStandings.json?limit=100")
    lists = data["MRData"]["StandingsTable"]["StandingsLists"]
    if not lists:
        return pd.DataFrame(columns=["driver_id", "driver_points_before", "driver_standing_before"])
    filas = [
        {
            "driver_id": d["Driver"]["driverId"],
            "driver_points_before": float(d["points"]),
            "driver_standing_before": int(d["position"]),
        }
        for d in lists[0]["DriverStandings"]
    ]
    return pd.DataFrame(filas)

# Ejemplo: standing ANTES de la ronda 5 de 2024 (o sea, tras la ronda 4)
standings_antes_r5 = driver_standings_before(2024, 5)
standings_antes_r5.head()


,driver_id,driver_points_before,driver_standing_before
0,max_verstappen,77.0,1
1,perez,64.0,2
2,leclerc,59.0,3
3,sainz,55.0,4
4,norris,37.0,5


## 5. Armar un historial de varias temporadas (para poder calcular rolling features)

Esto es lo más lento del prototipo porque son muchas llamadas a la API —
en Colab está bien, en el DAG esto es exactamente lo que va a vivir en la
capa de bronce para no repetirse.

Empiecen con 2-3 temporadas para prototipar rápido; para el dataset final
son las ~250 carreras que mencionaron en la propuesta.


In [14]:
def calendario(season: int) -> list[dict]:
    data = fetch_json(f"{season}.json?limit=100")
    return data["MRData"]["RaceTable"]["Races"]

def construir_historial(temporadas: list[int]) -> pd.DataFrame:
    todas_las_filas = []
    for season in temporadas:
        rondas = calendario(season)
        for carrera in rondas:
            round_ = int(carrera["round"])
            try:
                resultados = fetch_json(f"{season}/{round_}/results.json?limit=100")
            except RuntimeError as e:
                print(f"salteando {season}-{round_}: {e}")
                continue
            todas_las_filas.extend(resultados_a_filas(resultados))
            time.sleep(0.3)  # no golpear la API de más
    return pd.DataFrame(todas_las_filas)

# Prototipo chico: 2022-2024. Para el dataset final, extiendan el rango.
historial = construir_historial([2010,2011,2012,2013,2014,2015,2016,2027,2018,2019,2020,2021,2022, 2023, 2024,2025,2026])
historial.shape


(6775, 10)

## 6. Rolling features SIN leakage

La regla de oro: para calcular una feature de la fila `(season, round,
driver_id)`, solo se puede mirar `historial` filtrado a carreras
estrictamente anteriores a esa fecha. Por eso se recorre el historial ya
ordenado cronológicamente y se va acumulando "lo visto hasta ahora".


In [15]:
historial = historial.sort_values(["season", "round"]).reset_index(drop=True)

def avg_finish_last_n(historial_previo: pd.DataFrame, driver_id: str, n: int):
    previas = historial_previo[historial_previo["driver_id"] == driver_id].tail(n)
    if previas.empty:
        return None
    return previas["final_position"].mean()

def dnf_rate_last_n(historial_previo: pd.DataFrame, driver_id: str, n: int):
    previas = historial_previo[historial_previo["driver_id"] == driver_id].tail(n)
    if previas.empty:
        return None
    return (previas["status"] != "Finished").mean()

def avg_finish_at_circuit(historial_previo: pd.DataFrame, driver_id: str, circuit_id: str):
    previas = historial_previo[
        (historial_previo["driver_id"] == driver_id) & (historial_previo["circuit_id"] == circuit_id)
    ]
    if previas.empty:
        return None, 0
    return previas["final_position"].mean(), len(previas)


In [ ]:
def construir_dataset(historial: pd.DataFrame) -> pd.DataFrame:
    filas_finales = []

    # Ordenado cronológicamente: no cambia el resultado (el filtro de
    # "previo" ya es por fecha, no por orden de loop), pero hace que los
    # avisos de progreso y de standings faltantes se lean en orden y sea
    # más fácil ubicar dónde está parado si vuelve a cortarse.
    claves = (
        historial[["season", "round"]]
        .drop_duplicates()
        .sort_values(["season", "round"])
        .values.tolist()
    )

    total = len(claves)
    for i, (season, round_) in enumerate(claves, start=1):
        fecha_actual = historial[(historial["season"] == season) & (historial["round"] == round_)]
        # todo lo estrictamente anterior a esta fecha
        previo = historial[
            (historial["season"] < season) |
            ((historial["season"] == season) & (historial["round"] < round_))
        ]

        # driver_standings_before ya es tolerante a entradas sin "position"
        # (las descarta con aviso), pero si la API falla del todo tras los
        # reintentos, seguimos con standings vacío en vez de tirar abajo
        # las demás 250 carreras por una que no respondió.
        try:
            standings = driver_standings_before(season, round_)
        except Exception as e:
            print(f"  aviso: no se pudo bajar standings de {season}-{round_} ({e}); se sigue sin ese dato")
            standings = pd.DataFrame(columns=["driver_id", "driver_points_before", "driver_standing_before"])

        for _, fila in fecha_actual.iterrows():
            avg5 = avg_finish_last_n(previo, fila["driver_id"], 5)
            dnf10 = dnf_rate_last_n(previo, fila["driver_id"], 10)
            avg_circ, n_circ = avg_finish_at_circuit(previo, fila["driver_id"], fila["circuit_id"])

            standing_piloto = standings[standings["driver_id"] == fila["driver_id"]]
            puntos_antes = standing_piloto["driver_points_before"].iloc[0] if not standing_piloto.empty else 0.0
            pos_antes = standing_piloto["driver_standing_before"].iloc[0] if not standing_piloto.empty else None

            filas_finales.append({
                **fila.to_dict(),
                "driver_points_before": puntos_antes,
                "driver_standing_before": pos_antes,
                "driver_avg_finish_last5": avg5,
                "driver_dnf_rate_last10": dnf10,
                "driver_avg_finish_circuit": avg_circ,
                "driver_races_at_circuit": n_circ,
                "finished_top10": bool(fila["final_position"] is not None and fila["final_position"] <= 10),
            })

        if i % 10 == 0 or i == total:
            print(f"  {i}/{total} carreras procesadas")

    return pd.DataFrame(filas_finales)

dataset = construir_dataset(historial)
dataset.shape

  aviso: no se pudo bajar standings de 2010-2 ('position'); se sigue sin ese dato
  aviso: no se pudo bajar standings de 2010-3 ('position'); se sigue sin ese dato
  aviso: no se pudo bajar standings de 2010-4 ('position'); se sigue sin ese dato
  aviso: no se pudo bajar standings de 2010-5 ('position'); se sigue sin ese dato
  10/321 carreras procesadas
  aviso: no se pudo bajar standings de 2010-16 ('position'); se sigue sin ese dato
  aviso: no se pudo bajar standings de 2010-17 ('position'); se sigue sin ese dato
  aviso: no se pudo bajar standings de 2010-18 ('position'); se sigue sin ese dato
  20/321 carreras procesadas
  aviso: no se pudo bajar standings de 2011-2 ('position'); se sigue sin ese dato
  aviso: no se pudo bajar standings de 2011-3 ('position'); se sigue sin ese dato
  30/321 carreras procesadas
  aviso: no se pudo bajar standings de 2012-2 ('position'); se sigue sin ese dato
  40/321 carreras procesadas
  aviso: no se pudo bajar standings de 2012-3 ('position'); s

## 7. Los 7 criterios de calidad (los mismos de la consigna)

Corran esto antes de dar por bueno el dataset. Es literalmente el
checklist que les van a pedir defender en la entrega.


In [ ]:
def chequear_calidad(df: pd.DataFrame, clave=("season", "round", "driver_id")):
    print("== 1. Clave sin duplicados ==")
    duplicados = df.duplicated(subset=list(clave)).sum()
    print(f"  {'OK' if duplicados == 0 else 'FALLA'} -> {duplicados} filas duplicadas en {clave}")

    print("== 2. Volumen suficiente (> 1.000 filas) ==")
    print(f"  {'OK' if len(df) > 1000 else 'FALLA (esperado en el prototipo chico)'} -> {len(df)} filas")

    print("== 3. Ancho suficiente (>= 5 columnas útiles) ==")
    print(f"  {'OK' if df.shape[1] >= 5 else 'FALLA'} -> {df.shape[1]} columnas")

    print("== 4. Mezcla de tipos ==")
    print(df.dtypes.value_counts())

    print("== 5. Nulos conocidos ==")
    print(df.isna().mean().sort_values(ascending=False).head(10))

    print("== 6. Sin columnas 100% nulas ==")
    vacias = df.columns[df.isna().all()].tolist()
    print(f"  {'OK' if not vacias else 'FALLA'} -> columnas vacías: {vacias}")

    print("== Objetivo: distribución de finished_top10 ==")
    print(df["finished_top10"].value_counts(normalize=True))

chequear_calidad(dataset)


## 8. Próximos pasos

1. Mirar los nulos de `driver_avg_finish_circuit` (van a ser muchos en un
   prototipo de solo 3 temporadas — es esperable, es la primera vez que
   la mayoría de los pilotos corre cada circuito dentro de esta ventana).
2. Decidir la política para DNF/DNS en `final_position` (¿nulo? ¿última
   posición + 1? tiene que quedar documentada).
3. Fijar la decisión Opción A / Opción B sobre `grid_position`.
4. Extender `construir_historial` a las ~250 carreras del alcance final.
5. Migrar `fetch_json`, `resultados_a_filas`, `driver_standings_before` y
   las funciones de rolling features a `f1/jolpica.py`, `f1/transform.py`
   y `f1/features.py` del repositorio, tal cual están acá pero con types
   y sin el `time.sleep` artesanal (eso lo maneja Airflow con retries).
6. Recién ahí armar `dags/f1_ingest.py` con la capa de bronce (JSON crudo
   guardado en disco, particionado por season/round) y la capa de plata
   (este mismo `construir_dataset`, pero leyendo del bronce en vez de la
   red).
